In [1]:
import time, json, os
from pathlib import Path
import requests
import xmltodict
import pandas as pd
from tqdm import tqdm

In [ ]:
CACHE_DIR = Path("data/bgg_cache")
CACHE_DIR.mkdir(parents=True, exist_ok=True)

def fetch_with_retry(url, params=None, retries=5, backoff=1.0):
    for i in range(retries):
        try:
            r = requests.get(url, params=params, timeout=30)
            if r.status_code == 200:
                return r.text
            # 202 means "not ready" for some BGG endpoints: wait and retry
            if r.status_code in (202, 503):
                time.sleep(backoff * (i+1))
                continue
            r.raise_for_status()
        except requests.RequestException:
            time.sleep(backoff * (i+1))
    raise RuntimeError(f"Failed to fetch {url} after {retries} retries")

def get_game_xml(game_id):
    cache_file = CACHE_DIR / f"{game_id}.json"
    if cache_file.exists():
        return json.loads(cache_file.read_text())
    base = "https://www.boardgamegeek.com/xmlapi2/thing"
    xml = fetch_with_retry(base, params={"id": str(game_id), "stats": 1})
    doc = xmltodict.parse(xml)
    cache_file.write_text(json.dumps(doc))
    time.sleep(1.2)  # polite pause (tune as needed)
    return doc

def normalize_game(doc):
    # doc structure: {'items': {'item': {...}}}
    item = doc.get("items", {}).get("item")
    if item is None:
        return None
    gid = item.get("@id")
    name = None
    for n in item.get("name", [] if isinstance(item.get("name"), list) else [item.get("name")]):
        if n.get("@type") == "primary":
            name = n.get("@value")
    year = item.get("yearpublished", {}).get("@value")
    players_min = item.get("minplayers", {}).get("@value")
    players_max = item.get("maxplayers", {}).get("@value")
    playingtime = item.get("playingtime", {}).get("@value")
    avg = item.get("statistics", {}).get("ratings", {}).get("average", {}).get("@value")
    users_rated = item.get("statistics", {}).get("ratings", {}).get("usersrated", {}).get("@value")
    desc = item.get("description")
    return {
        "id": int(gid) if gid else None,
        "name": name,
        "year": int(year) if year and year.isdigit() else None,
        "minplayers": int(players_min) if players_min else None,
        "maxplayers": int(players_max) if players_max else None,
        "playingtime": int(playingtime) if playingtime else None,
        "rating_avg": float(avg) if avg and avg != "N/A" else None,
        "users_rated": int(users_rated) if users_rated else None,
        "description": desc
    }

def fetch_multiple(ids):
    rows = []
    for gid in tqdm(ids):
        doc = get_game_xml(gid)
        rec = normalize_game(doc)
        if rec:
            rows.append(rec)
    return pd.DataFrame(rows)

In [ ]:
# Uso ejemplo: obtener lista de IDs manualmente o desde /xmlapi2/hot o /xmlapi2/collection
sample_ids = [13, 174430, 68448]  # reemplazar con IDs reales o generar programáticamente
df = fetch_multiple(sample_ids)
df.to_parquet("data/bgg_sample.parquet", index=False)
print(df.head())

In [ ]:
# base_url_bgg = 'https://www.boardgamegeek.com/'
# endpoint = 'xmlapi2/thing'
# params = {
#     'id': '13'
# }
# response = requests.get(base_url_bgg + endpoint, params=params)
# print(response.status_code)
# print(response.text)


401
Unauthorized. See https://boardgamegeek.com/using_the_xml_api


## Sondeo de endpoints — api.valentialudica.com

Esta celda contiene un script para sondear rutas comunes del API público `api.valentialudica.com`. 
Ejecuta la siguiente celda de código para obtener códigos HTTP y fragmentos de respuesta.

In [3]:
# Probe API endpoints for api.valentialudica.com
import requests
paths = ['/', '/openapi.json', '/swagger.json', '/v1/openapi.json', '/v1/swagger.json',
         '/health', '/status', '/api', '/docs', '/.well-known/openid-configuration',
         '/v1', '/v2', '/products', '/games', '/customers', '/orders', '/auth', '/login']
for p in paths:
    url = 'https://api.valentialudica.com' + p
    print('===', url, '===')
    try:
        r = requests.get(url, timeout=10)
        print('Status:', r.status_code)
        text = r.text or ''
        print(text[:2000] + ('\n...(truncated)' if len(text) > 2000 else ''))
    except Exception as e:
        print('Error:', e)

=== https://api.valentialudica.com/ ===
Status: 404
{"message":"Route GET:/ not found","error":"Not Found","statusCode":404}
=== https://api.valentialudica.com/openapi.json ===
Status: 404
{"message":"Route GET:/openapi.json not found","error":"Not Found","statusCode":404}
=== https://api.valentialudica.com/swagger.json ===
Status: 404
{"message":"Route GET:/swagger.json not found","error":"Not Found","statusCode":404}
=== https://api.valentialudica.com/v1/openapi.json ===
Status: 404
{"message":"Route GET:/v1/openapi.json not found","error":"Not Found","statusCode":404}
=== https://api.valentialudica.com/v1/swagger.json ===
Status: 404
{"message":"Route GET:/v1/swagger.json not found","error":"Not Found","statusCode":404}
=== https://api.valentialudica.com/health ===
Status: 200
{"status":"ok"}
=== https://api.valentialudica.com/status ===
Status: 404
{"message":"Route GET:/status not found","error":"Not Found","statusCode":404}
=== https://api.valentialudica.com/api ===
Status: 404
{

### Sondeos adicionales automáticos

La siguiente celda realiza peticiones `GET` y `OPTIONS` a rutas adicionales comunes, además de intentar acceder a `openapi.yaml`, `api-docs`, `swagger-ui` y `robots.txt`.
Ejecuta la celda para recoger más información pública del API y guarda los resultados en `data/api_probe_results.json`.

In [4]:
import requests, json, time
from pathlib import Path
OUT = Path('data/api_probe_results.json')
OUT.parent.mkdir(parents=True, exist_ok=True)
paths = [
    '/openapi.yaml', '/openapi.yml', '/api-docs', '/swagger-ui', '/swagger-ui.html',
    '/swagger', '/api/docs', '/docs/swagger', '/graphql', '/graphql/schema',
    '/.well-known/jwks.json', '/.well-known/assetlinks.json', '/robots.txt'
]
results = []
for p in paths:
    url = 'https://api.valentialudica.com' + p
    entry = {'path': p, 'url': url, 'get': None, 'options': None}
    try:
        r = requests.get(url, timeout=10)
        entry['get'] = {'status': r.status_code, 'content_snippet': (r.text[:2000] if r.text else '')}
    except Exception as e:
        entry['get'] = {'error': str(e)}
    try:
        r2 = requests.options(url, timeout=10)
        entry['options'] = {'status': getattr(r2, 'status_code', None), 'allow': r2.headers.get('Allow') if r2.headers else None}
    except Exception as e:
        entry['options'] = {'error': str(e)}
    results.append(entry)
    time.sleep(0.2)
OUT.write_text(json.dumps({'checked_at': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()), 'results': results}, ensure_ascii=False, indent=2))
print('Saved probe results to', OUT)

Saved probe results to data\api_probe_results.json


Una vez ejecutadas las celdas, pega aquí el contenido de `data/api_probe_results.json` o sube el fichero para que genere el informe Markdown con los endpoints detectados.

In [5]:
import json
from pathlib import Path
import requests

summary_path = Path('data/api_probe_results.json')
if summary_path.exists():
    data = json.loads(summary_path.read_text(encoding='utf-8'))
    results = data.get('results', [])
    ok = [r for r in results if isinstance(r.get('get'), dict) and r['get'].get('status') == 200]
    not_found = [r for r in results if isinstance(r.get('get'), dict) and r['get'].get('status') == 404]
    print(f"Loaded {len(results)} probe entries from {summary_path}")
    print(f"GET 200: {len(ok)}")
    print(f"GET 404: {len(not_found)}")
    for item in ok:
        print(f"200 -> {item['path']}")
else:
    print(f"Missing {summary_path}; run the probe cell first.")

for method in ('GET', 'HEAD', 'OPTIONS'):
    try:
        response = requests.request(method, 'https://api.valentialudica.com/health', timeout=10)
        allow = response.headers.get('Allow')
        print(f"health {method}: {response.status_code} allow={allow}")
        if method == 'GET':
            print(response.text)
    except Exception as exc:
        print(f"health {method}: error={exc}")

Loaded 13 probe entries from data\api_probe_results.json
GET 200: 1
GET 404: 12
200 -> /robots.txt
health GET: 200 allow=None
{"status":"ok"}
health HEAD: 200 allow=None
health OPTIONS: 400 allow=None


### Pruebas autenticadas

Esta celda usa un token Bearer desde la variable de entorno `VALENTIA_ACCESS_TOKEN` para probar rutas candidatas protegidas. No imprime el token y guarda un resumen de estados HTTP.

In [ ]:
import os, json, time
from pathlib import Path
import requests

token = os.environ.get('VALENTIA_ACCESS_TOKEN')
if not token:
    raise RuntimeError('Set VALENTIA_ACCESS_TOKEN in your environment before running this cell')

headers = {'Authorization': f'Bearer {token}'}
auth_hosts = ['https://api.valentialudica.com', 'https://app.valentialudica.com']
candidates = [
    '/me', '/profile', '/user', '/users/me', '/api/me', '/api/user', '/account',
    '/auth/me', '/auth/profile', '/session', '/sessions', '/dashboard',
    '/notifications', '/settings', '/members/me', '/graphql'
]
summary = []
for host in auth_hosts:
    for path in candidates:
        url = host + path
        row = {'host': host, 'path': path, 'url': url}
        try:
            r = requests.get(url, headers=headers, timeout=10)
            row['get'] = {'status': r.status_code, 'content_type': r.headers.get('Content-Type'), 'snippet': (r.text[:500] if r.text else '')}
        except Exception as exc:
            row['get'] = {'error': str(exc)}
        try:
            r2 = requests.options(url, headers=headers, timeout=10)
            row['options'] = {'status': r2.status_code, 'allow': r2.headers.get('Allow')}
        except Exception as exc:
            row['options'] = {'error': str(exc)}
        summary.append(row)
        time.sleep(0.15)

out = Path('data/auth_probe_results.json')
out.parent.mkdir(parents=True, exist_ok=True)
out.write_text(json.dumps({'checked_at': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()), 'results': summary}, ensure_ascii=False, indent=2))
print('Saved auth probe results to', out)
for row in summary:
    g = row.get('get', {})
    if g.get('status') in (200, 401, 403):
        print(f"{row['url']} -> GET {g.get('status')}")

Saved auth probe results to data\auth_probe_results.json
https://api.valentialudica.com/auth/me -> GET 200
https://app.valentialudica.com/me -> GET 200
https://app.valentialudica.com/profile -> GET 200
https://app.valentialudica.com/user -> GET 200
https://app.valentialudica.com/users/me -> GET 200
https://app.valentialudica.com/api/me -> GET 200
https://app.valentialudica.com/api/user -> GET 200
https://app.valentialudica.com/account -> GET 200
https://app.valentialudica.com/auth/me -> GET 200
https://app.valentialudica.com/auth/profile -> GET 200
https://app.valentialudica.com/session -> GET 200
https://app.valentialudica.com/sessions -> GET 200
https://app.valentialudica.com/dashboard -> GET 200
https://app.valentialudica.com/notifications -> GET 200
https://app.valentialudica.com/settings -> GET 200
https://app.valentialudica.com/members/me -> GET 200
https://app.valentialudica.com/graphql -> GET 200


Si esta celda encuentra respuestas distintas de 404/401, ya tenemos una pista clara de rutas internas. Si quieres, luego te preparo otra celda para probar `POST` en `/graphql` o rutas de login con el mismo token.

In [10]:
import os
import json
from pathlib import Path
import requests

board_games_token = os.environ.get('VALENTIA_ACCESS_TOKEN')
if not board_games_token:
    raise RuntimeError('Set VALENTIA_ACCESS_TOKEN in the notebook kernel before running this cell')

headers = {'Authorization': f'Bearer {board_games_token}'}
urls = [
    'https://api.valentialudica.com/board-games',
    'https://api.valentialudica.com/board-games?limit=5',
    'https://app.valentialudica.com/board-games',
]
results = []
for url in urls:
    row = {'url': url}
    for method in ('GET', 'HEAD', 'OPTIONS'):
        try:
            response = requests.request(method, url, headers=headers, timeout=15)
            row[method.lower()] = {
                'status': response.status_code,
                'content_type': response.headers.get('Content-Type'),
                'allow': response.headers.get('Allow'),
                'snippet': response.text[:800] if response.text else '',
            }
        except Exception as exc:
            row[method.lower()] = {'error': str(exc)}
    results.append(row)

out = Path('data/board_games_probe_results.json')
out.parent.mkdir(parents=True, exist_ok=True)
out.write_text(json.dumps({'checked_at': __import__('time').strftime('%Y-%m-%dT%H:%M:%SZ', __import__('time').gmtime()), 'results': results}, ensure_ascii=False, indent=2))
print('Saved board-games probe results to', out)
for row in results:
    print(row['url'])
    for method in ('get', 'head', 'options'):
        payload = row.get(method, {})
        print(f"  {method.upper()}: {payload.get('status')} {payload.get('content_type') or ''}")

Saved board-games probe results to data\board_games_probe_results.json
https://api.valentialudica.com/board-games
  GET: 200 application/json; charset=utf-8
  HEAD: 200 application/json; charset=utf-8
  OPTIONS: 400 text/plain
https://api.valentialudica.com/board-games?limit=5
  GET: 200 application/json; charset=utf-8
  HEAD: 200 application/json; charset=utf-8
  OPTIONS: 400 text/plain
https://app.valentialudica.com/board-games
  GET: 200 text/html
  HEAD: 200 text/html
  OPTIONS: 405 text/html


In [14]:
import os
import json
from pathlib import Path
import pandas as pd
import requests

# Reuse an existing token from the notebook if available; otherwise fall back to env.
token_value = globals().get('board_games_token') or globals().get('token') or globals().get('VALENTIA_ACCESS_TOKEN') or os.environ.get('VALENTIA_ACCESS_TOKEN')
if not token_value:
    raise RuntimeError('No token available. Set VALENTIA_ACCESS_TOKEN or define board_games_token in the notebook kernel.')

url = 'https://api.valentialudica.com/board-games'
headers = {'Authorization': f'Bearer {token_value}'}
response = requests.get(url, headers=headers, timeout=20)
response.raise_for_status()
payload = response.json()

board_games = payload.get('boardGames', [])
if not board_games:
    raise RuntimeError('The endpoint returned no boardGames items')

# Save the raw API payload.
raw_out = Path('data/board_games_api_response.json')
raw_out.parent.mkdir(parents=True, exist_ok=True)
raw_out.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding='utf-8')

# Also keep a clean JSON array with only the boardGames items.
items_out = Path('data/board_games_raw.json')
items_out.write_text(json.dumps(board_games, ensure_ascii=False, indent=2), encoding='utf-8')

# Flatten nested details into a reviewable DataFrame.
df_board_games = pd.json_normalize(board_games, sep='_')

df_csv_out = Path('data/board_games_dataframe.csv')
df_parquet_out = Path('data/board_games_dataframe.parquet')
df_board_games.to_csv(df_csv_out, index=False, encoding='utf-8')

parquet_status = 'skipped'
try:
    df_board_games.to_parquet(df_parquet_out, index=False)
    parquet_status = 'written'
except Exception as exc:
    parquet_status = f'skipped: {exc.__class__.__name__}'

print(f'HTTP status: {response.status_code}')
print(f'Extracted {len(board_games)} board games')
print('Saved raw API payload to:', raw_out)
print('Saved boardGames array to:', items_out)
print('Saved DataFrame to:', df_csv_out)
print('Parquet:', parquet_status)
if parquet_status == 'written':
    print('Saved DataFrame to:', df_parquet_out)
print('\nColumns:')
print(list(df_board_games.columns))
print('\nHead:')
print(df_board_games.head(10).to_string(index=False))

HTTP status: 200
Extracted 742 board games
Saved raw API payload to: data\board_games_api_response.json
Saved boardGames array to: data\board_games_raw.json
Saved DataFrame to: data\board_games_dataframe.csv
Parquet: skipped: ImportError

Columns:
['name', 'original_name', 'thumbnail', 'image', 'id', 'comment', 'year_published', 'details_min_players', 'details_max_players', 'details_average_rating', 'details_average_weight', 'details_playing_time', 'details_num_ratings']

Head:
                      name  original_name                                                                                                                                     thumbnail                                                                                                                                  image                                   id                                                    comment  year_published  details_min_players  details_max_players  details_average_rating  details_average_wei

In [22]:
import pandas as pd
from pathlib import Path

# Cargar el CSV actual
csv_input = Path('data/board_games_dataframe.csv')
df_full = pd.read_csv(csv_input, encoding='utf-8')

# Identificar columnas con URLs para eliminar
columns_to_drop = ['thumbnail', 'image']
columns_to_drop = [col for col in columns_to_drop if col in df_full.columns]

# Crear DataFrame limpio sin las columnas de enlaces
df_clean = df_full.drop(columns=columns_to_drop)

# Verificar si la columna 'comment' existe
if 'comment' in df_clean.columns:
    # Reemplazar saltos de línea (\n, \r\n, \r) por espacios en la columna 'comment'
    df_clean['comment'] = df_clean['comment'].str.replace('\r\n', '. ', regex=False)
    df_clean['comment'] = df_clean['comment'].str.replace('\n', '. ', regex=False)
    df_clean['comment'] = df_clean['comment'].str.replace('\r', '. ', regex=False)
    
    # Limpiar espacios múltiples consecutivos por un solo espacio
    df_clean['comment'] = df_clean['comment'].str.replace(r'\s+', ' ', regex=True).str.strip()

    print('✓ Saltos de línea eliminados de la columna "comment"')
    print(f'✓ Archivo guardado: {csv_input}')
    print('\nMuestra de los comentarios limpios:')
    print(df_clean[['name', 'comment']].head(10).to_string(index=False))
else:
    print('⚠ La columna "comment" no existe en el CSV')

# Guardar el CSV limpio
csv_output = Path('data/board_games_clean.csv')
df_clean.to_csv(csv_output, index=False, encoding='utf-8')

print(f'✓ Cargado CSV original: {len(df_full)} filas, {len(df_full.columns)} columnas')
print(f'✓ Eliminadas columnas: {columns_to_drop}')
print(f'✓ CSV limpio guardado: {csv_output} ({len(df_clean)} filas, {len(df_clean.columns)} columnas)')
print('\nColumnas restantes:')
print(list(df_clean.columns))
print('\nDatos:')
print(df_clean.head(10).to_string(index=False))

✓ Saltos de línea eliminados de la columna "comment"
✓ Archivo guardado: data\board_games_dataframe.csv

Muestra de los comentarios limpios:
                      name                                                    comment
Los Descubridores de Catán                                     Pendiente de revisión.
               Carcassonne                                     Pendiente de revisión.
                  Pandemic            Cedido por: Jaime, Tato. Pendiente de revisión.
         Terraforming Mars                                     Pendiente de revisión.
                 7 Wonders                                     Pendiente de revisión.
                  Wingspan                                     Pendiente de revisión.
                      Azul                                     Pendiente de revisión.
            Código secreto                                                    Cedido.
     ¡Aventureros al Tren! En caja de Alta Tensión en alemán.. Pendiente de revisión.

## Extracción del endpoint `/role-games`

Similar a board-games, vamos a probar, extraer y limpiar los datos del endpoint `/role-games`.


In [16]:
import os
import json
from pathlib import Path
import requests

role_games_token = os.environ.get('VALENTIA_ACCESS_TOKEN') or globals().get('board_games_token') or globals().get('token')
if not role_games_token:
    raise RuntimeError('Set VALENTIA_ACCESS_TOKEN in the notebook kernel before running this cell')

headers = {'Authorization': f'Bearer {role_games_token}'}
urls = [
    'https://api.valentialudica.com/role-games',
    'https://api.valentialudica.com/role-games?limit=5',
    'https://app.valentialudica.com/role-games',
]
results = []
for url in urls:
    row = {'url': url}
    for method in ('GET', 'HEAD', 'OPTIONS'):
        try:
            response = requests.request(method, url, headers=headers, timeout=15)
            row[method.lower()] = {
                'status': response.status_code,
                'content_type': response.headers.get('Content-Type'),
                'allow': response.headers.get('Allow'),
                'snippet': response.text[:800] if response.text else '',
            }
        except Exception as exc:
            row[method.lower()] = {'error': str(exc)}
    results.append(row)

out = Path('data/role_games_probe_results.json')
out.parent.mkdir(parents=True, exist_ok=True)
out.write_text(json.dumps({'checked_at': __import__('time').strftime('%Y-%m-%dT%H:%M:%SZ', __import__('time').gmtime()), 'results': results}, ensure_ascii=False, indent=2))
print('Saved role-games probe results to', out)
for row in results:
    print(row['url'])
    for method in ('get', 'head', 'options'):
        payload = row.get(method, {})
        print(f"  {method.upper()}: {payload.get('status')} {payload.get('content_type') or ''}")

Saved role-games probe results to data\role_games_probe_results.json
https://api.valentialudica.com/role-games
  GET: 200 application/json; charset=utf-8
  HEAD: 200 application/json; charset=utf-8
  OPTIONS: 400 text/plain
https://api.valentialudica.com/role-games?limit=5
  GET: 200 application/json; charset=utf-8
  HEAD: 200 application/json; charset=utf-8
  OPTIONS: 400 text/plain
https://app.valentialudica.com/role-games
  GET: 200 text/html
  HEAD: 200 text/html
  OPTIONS: 405 text/html


In [18]:
import os
import json
from pathlib import Path
import pandas as pd
import requests

# Reuse an existing token from the notebook if available; otherwise fall back to env.
token_value = globals().get('role_games_token') or globals().get('board_games_token') or globals().get('token') or globals().get('VALENTIA_ACCESS_TOKEN') or os.environ.get('VALENTIA_ACCESS_TOKEN')
if not token_value:
    raise RuntimeError('No token available. Set VALENTIA_ACCESS_TOKEN or define role_games_token in the notebook kernel.')

url = 'https://api.valentialudica.com/role-games'
headers = {'Authorization': f'Bearer {token_value}'}
response = requests.get(url, headers=headers, timeout=20)
response.raise_for_status()
payload = response.json()

# Handle both wrapped {'roleGames': [...]} and direct list [...] responses
if isinstance(payload, dict):
    role_games = payload.get('roleGames', payload.get('roleGames', []))
else:
    role_games = payload if isinstance(payload, list) else []

if not role_games:
    raise RuntimeError('The endpoint returned no roleGames items')

# Save the raw API payload.
raw_out = Path('data/role_games_api_response.json')
raw_out.parent.mkdir(parents=True, exist_ok=True)
raw_out.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding='utf-8')

# Also keep a clean JSON array with only the roleGames items.
items_out = Path('data/role_games_raw.json')
items_out.write_text(json.dumps(role_games, ensure_ascii=False, indent=2), encoding='utf-8')

# Flatten nested details into a reviewable DataFrame.
df_role_games = pd.json_normalize(role_games, sep='_')

df_csv_out = Path('data/role_games_dataframe.csv')
df_parquet_out = Path('data/role_games_dataframe.parquet')
df_role_games.to_csv(df_csv_out, index=False, encoding='utf-8')

parquet_status = 'skipped'
try:
    df_role_games.to_parquet(df_parquet_out, index=False)
    parquet_status = 'written'
except Exception as exc:
    parquet_status = f'skipped: {exc.__class__.__name__}'

print(f'HTTP status: {response.status_code}')
print(f'Extracted {len(role_games)} role games')
print('Saved raw API payload to:', raw_out)
print('Saved roleGames array to:', items_out)
print('Saved DataFrame to:', df_csv_out)
print('Parquet:', parquet_status)
if parquet_status == 'written':
    print('Saved DataFrame to:', df_parquet_out)
print('\nColumns:')
print(list(df_role_games.columns))
print('\nHead:')
print(df_role_games.head(10).to_string(index=False))

HTTP status: 200
Extracted 624 role games
Saved raw API payload to: data\role_games_api_response.json
Saved roleGames array to: data\role_games_raw.json
Saved DataFrame to: data\role_games_dataframe.csv
Parquet: skipped: ImportError

Columns:
['id', 'name', 'has_image', 'publisher', 'language', 'image_url', 'image_public_url']

Head:
                                  id                                             name  has_image         publisher language image_url image_public_url
df17ef6a-6445-4f1b-84ee-40b044be86e4                           7th Sea Player's Guide      False Factoria de ideas  Español      None              NaN
8af7f672-7a95-4b4c-a8b4-bd96aa9fb1af                                           7º Mar      False         Nosolorol  Español      None              NaN
ea90611b-c972-445e-a922-345a2d71a48f                                7º Mar - Castilla      False Factoria de ideas  Español      None              NaN
2e43c081-4419-4e06-a0fc-6893012300fe                        

In [19]:
import pandas as pd
from pathlib import Path

# Cargar el CSV actual
csv_input = Path('data/role_games_dataframe.csv')
df_full = pd.read_csv(csv_input, encoding='utf-8')

# Identificar columnas con URLs para eliminar
columns_to_drop = ['thumbnail', 'image']
columns_to_drop = [col for col in columns_to_drop if col in df_full.columns]

# Crear DataFrame limpio sin las columnas de enlaces
df_clean = df_full.drop(columns=columns_to_drop)

# Guardar el CSV limpio
csv_output = Path('data/role_games_clean.csv')
df_clean.to_csv(csv_output, index=False, encoding='utf-8')

print(f'✓ Cargado CSV original: {len(df_full)} filas, {len(df_full.columns)} columnas')
print(f'✓ Eliminadas columnas: {columns_to_drop}')
print(f'✓ CSV limpio guardado: {csv_output} ({len(df_clean)} filas, {len(df_clean.columns)} columnas)')
print('\nColumnas restantes:')
print(list(df_clean.columns))
print('\nDatos:')
print(df_clean.head(10).to_string(index=False))

✓ Cargado CSV original: 624 filas, 7 columnas
✓ Eliminadas columnas: []
✓ CSV limpio guardado: data\role_games_clean.csv (624 filas, 7 columnas)

Columnas restantes:
['id', 'name', 'has_image', 'publisher', 'language', 'image_url', 'image_public_url']

Datos:
                                  id                                             name  has_image         publisher language image_url image_public_url
df17ef6a-6445-4f1b-84ee-40b044be86e4                           7th Sea Player's Guide      False Factoria de ideas  Español       NaN              NaN
8af7f672-7a95-4b4c-a8b4-bd96aa9fb1af                                           7º Mar      False         Nosolorol  Español       NaN              NaN
ea90611b-c972-445e-a922-345a2d71a48f                                7º Mar - Castilla      False Factoria de ideas  Español       NaN              NaN
2e43c081-4419-4e06-a0fc-6893012300fe                              8 Semanas más tarde      False      Holocubierta  Español       NaN   

## Limpieza de saltos de línea en la columna "comment"

Eliminar saltos de línea en los comentarios de board-games para que el CSV sea más legible y procesable.
